# 04.2 — Speech solutions lab

**No microphone required.** The lab synthesizes its own audio into `lab_output/`
first, then recognizes from those files. If you do have a microphone, every
recognition cell has a one-line change in a comment to use it instead.

**Prerequisites**
- `.env` with `AZURE_SPEECH_REGION` (and `AZURE_SPEECH_KEY` as a fallback)
- Role **Cognitive Services Speech User** on the Foundry resource
- `pip install -r requirements.txt`
- Optional: a `gpt-4o-mini-audio-preview` deployment, recorded in `.env` as
  `MODEL_AUDIO`. Sections 6 and 8 skip cleanly without it.

Cost: seconds of audio through Speech, plus a few audio-token calls. Under $2.

## 1. Setup and authentication

Speech is the one service in this course where a key is defensible, so it is worth
being precise about what is actually supported:

| Object | Entra ID path |
|---|---|
| `SpeechRecognizer`, `ConversationTranscriber` | `SpeechConfig(token_credential=..., endpoint=<custom domain>)` |
| `TranslationRecognizer` | `SpeechTranslationConfig(token_credential=..., endpoint=<custom domain>)` |
| `SpeechSynthesizer` and the rest | authorization token `aad#{resourceId}#{token}` |

The cell below builds all three and falls back to the key only if Entra ID is
unavailable — printing which path it took, so you always know.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client, show_usage

import base64, json, time
import azure.cognitiveservices.speech as speechsdk

OUT = pathlib.Path.cwd() / "lab_output"
OUT.mkdir(exist_ok=True)
(OUT / ".gitignore").write_text("*\n", encoding="utf-8")  # keep audio out of git

REGION = cfg.require("AZURE_SPEECH_REGION", unit="04.2")
RESOURCE = cfg.require("AZURE_AI_FOUNDRY_RESOURCE")
CUSTOM_DOMAIN_ENDPOINT = f"https://{RESOURCE}.cognitiveservices.azure.com/"
RESOURCE_ID = (
    f"/subscriptions/{cfg.require('AZURE_SUBSCRIPTION_ID')}"
    f"/resourceGroups/{cfg.require('AZURE_RESOURCE_GROUP')}"
    f"/providers/Microsoft.CognitiveServices/accounts/{RESOURCE}"
)

print("region   :", REGION)
print("endpoint :", CUSTOM_DOMAIN_ENDPOINT)
print("key set  :", bool(cfg.get("AZURE_SPEECH_KEY")))

In [ ]:
SCOPE = "https://cognitiveservices.azure.com/.default"


def aad_authorization_token():
    """The Speech authorization-token format: aad#{resourceId}#{entraAccessToken}.

    Both the 'aad#' prefix and the '#' separator are required. The token expires,
    so long-lived recognizers reassign speech_config.authorization_token instead of
    rebuilding the config.
    """
    token = credential().get_token(SCOPE).token
    return f"aad#{RESOURCE_ID}#{token}"


def speech_config(*, prefer_entra=True):
    """SpeechConfig for synthesis. Entra ID via authorization token, key as fallback."""
    if prefer_entra:
        try:
            cfg_ = speechsdk.SpeechConfig(auth_token=aad_authorization_token(), region=REGION)
            cfg_._ai103_auth = "entra (aad# authorization token)"
            return cfg_
        except Exception as exc:
            print("Entra ID path unavailable:", type(exc).__name__, exc)
    key = cfg.require("AZURE_SPEECH_KEY", unit="04.2")
    cfg_ = speechsdk.SpeechConfig(subscription=key, region=REGION)
    cfg_._ai103_auth = "key (fallback)"
    return cfg_


def recognizer_config():
    """SpeechConfig for recognition. SpeechRecognizer accepts a TokenCredential
    directly, but only with a custom-domain endpoint."""
    try:
        cfg_ = speechsdk.SpeechConfig(
            token_credential=credential(), endpoint=CUSTOM_DOMAIN_ENDPOINT
        )
        cfg_._ai103_auth = "entra (token_credential + custom domain)"
        return cfg_
    except Exception as exc:
        print("token_credential path unavailable:", type(exc).__name__, exc)
        key = cfg.require("AZURE_SPEECH_KEY", unit="04.2")
        cfg_ = speechsdk.SpeechConfig(subscription=key, region=REGION)
        cfg_._ai103_auth = "key (fallback)"
        return cfg_


print("synthesis  :", speech_config()._ai103_auth)
print("recognition:", recognizer_config()._ai103_auth)

If the recognition path fell back to a key, the usual cause is that the resource has
no custom subdomain. Check and fix (note: **cannot be changed later**):

```powershell
az cognitiveservices account show -n $env:AZURE_AI_FOUNDRY_RESOURCE `
  -g $env:AZURE_RESOURCE_GROUP --query "properties.customSubDomainName"
```

## 2. Text to speech

Three output destinations, and the choice matters more than it looks:

| Destination | Config | Use |
|---|---|---|
| Speakers | `use_default_speaker=True` | Local demo |
| File | `filename=...` | Batch, testing, headless |
| Memory | `audio_config=None` | Servers — read `result.audio_data` |

Set the output format explicitly. The default is not always what your playback path
or a downstream model expects, and a sample-rate mismatch produces chipmunk audio
with no error anywhere.

In [ ]:
GREETING = (
    "Thank you for calling Northwind Support. Your order S O dash four four seven one "
    "shipped on Tuesday and should arrive by Friday."
)
PLAIN_WAV = OUT / "plain.wav"

sc = speech_config()
sc.speech_synthesis_voice_name = "en-GB-SoniaNeural"
sc.set_speech_synthesis_output_format(
    speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm
)

synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=sc,
    audio_config=speechsdk.audio.AudioOutputConfig(filename=str(PLAIN_WAV)),
)

start = time.perf_counter()
result = synthesizer.speak_text_async(GREETING).get()
elapsed = (time.perf_counter() - start) * 1000

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"wrote {PLAIN_WAV.name}  {PLAIN_WAV.stat().st_size:,} bytes  in {elapsed:.0f} ms")
    print(f"audio duration: {result.audio_duration.total_seconds():.2f} s")
else:
    print("reason:", result.reason)
    if result.reason == speechsdk.ResultReason.Canceled:
        d = result.cancellation_details
        print("  ", d.reason, d.error_details)

> **Exam note.** Notice that a failure does **not** raise. It comes back as
> `ResultReason.Canceled` with `cancellation_details`. Every Speech SDK example that
> ignores `reason` is a silent-failure bug waiting to happen — this is the single
> most important habit in this SDK.

### Streaming synthesis — the latency that matters for agents

`speak_text_async(...).get()` waits for the entire utterance. For a voice agent, the
number a user perceives is **time to first byte of audio**, not total synthesis
time. Pulling from an `AudioDataStream` lets playback start while the rest is still
being generated.

Watch the gap between the two timings below. On a long utterance it is the
difference between a responsive agent and a dead-air pause.

In [ ]:
LONG = (
    "Here is a summary of your account. " + 
    "You have three open tickets, two shipments in transit, and one invoice awaiting payment. "
    "The oldest ticket concerns a delayed router delivery to your Manchester office, "
    "and the assigned engineer has requested additional diagnostic information."
)

sc_stream = speech_config()
sc_stream.speech_synthesis_voice_name = "en-GB-SoniaNeural"
sc_stream.set_speech_synthesis_output_format(
    speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm
)

# audio_config=None -> nothing is written to a device or file; we pull the stream.
stream_synth = speechsdk.SpeechSynthesizer(speech_config=sc_stream, audio_config=None)

start = time.perf_counter()
result = stream_synth.start_speaking_text_async(LONG).get()
stream = speechsdk.AudioDataStream(result)

buffer = bytes(32000)
first_chunk_ms = None
total = 0
while True:
    n = stream.read_data(buffer)
    if n == 0:
        break
    if first_chunk_ms is None:
        first_chunk_ms = (time.perf_counter() - start) * 1000
    total += n
complete_ms = (time.perf_counter() - start) * 1000

print(f"first audio chunk : {first_chunk_ms:.0f} ms   <- what the user perceives")
print(f"full synthesis    : {complete_ms:.0f} ms")
print(f"bytes             : {total:,}")

## 3. SSML

Plain text read the order reference as a word. SSML fixes that and adds pacing,
pronunciation, and speaking style. Note the two namespaces — omitting `xmlns:mstts`
while using `<mstts:express-as>` gives you a `Canceled` result with
`Invalid SSML`, not a helpful exception.

In [ ]:
SSML = """<speak version="1.0"
       xmlns="http://www.w3.org/2001/10/synthesis"
       xmlns:mstts="https://www.w3.org/2001/mstts"
       xml:lang="en-GB">
  <voice name="en-GB-SoniaNeural">
    <mstts:express-as style="empathetic" styledegree="1.5">
      I am sorry your delivery was late.
    </mstts:express-as>
    <break time="400ms"/>
    Your order reference is
    <prosody rate="-20%">
      <say-as interpret-as="characters">SO</say-as>
      <break time="200ms"/>
      <say-as interpret-as="digits">4471</say-as>.
    </prosody>
    <break time="300ms"/>
    A replacement <phoneme alphabet="ipa" ph="&#x2C8;n&#x254;&#x2D0;&#x03B8;w&#x26A;nd">Northwind</phoneme>
    X200 will arrive by <say-as interpret-as="date" format="dmy">20-03-2026</say-as>.
  </voice>
</speak>"""

SSML_WAV = OUT / "ssml.wav"
sc = speech_config()
sc.set_speech_synthesis_output_format(
    speechsdk.SpeechSynthesisOutputFormat.Riff24Khz16BitMonoPcm
)
synth = speechsdk.SpeechSynthesizer(
    speech_config=sc,
    audio_config=speechsdk.audio.AudioOutputConfig(filename=str(SSML_WAV)),
)
result = synth.speak_ssml_async(SSML).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"wrote {SSML_WAV.name}  duration {result.audio_duration.total_seconds():.2f} s")
    print("compare with plain.wav: the reference is now spelled out and paced")
else:
    d = result.cancellation_details
    print("canceled:", d.reason, d.error_details)

If `express-as` failed, the voice does not support styles. Not every neural voice
does — the **Styles** badge in the Speech Studio voice gallery tells you which. The
synthesis still succeeds; the style is simply ignored, which is a quietly annoying
failure mode.

| Requirement | Element |
|---|---|
| Read `SO-4471` as characters and digits | `<say-as interpret-as="characters">` / `"digits"` |
| Pronounce a brand or drug name | `<phoneme alphabet="ipa">`, or `<lexicon>` for many terms |
| Sound sympathetic | `<mstts:express-as style="empathetic">` |
| Slow a confirmation down | `<prosody rate="-20%">` |
| Pause between list items | `<break time="..."/>` |
| Switch language mid-sentence | `<lang xml:lang="...">` on a multilingual voice |

## 4. Speech to text

Recognize from the file we just made. Swap `AudioConfig(filename=...)` for
`AudioConfig(use_default_microphone=True)` if you have a microphone.

`recognize_once_async` stops at the **first end-of-speech or 15 seconds**, whichever
comes first. On a longer recording it returns the first utterance and reports
success — a silent truncation that is easy to miss.

In [ ]:
rc = recognizer_config()
rc.speech_recognition_language = "en-GB"

recognizer = speechsdk.SpeechRecognizer(
    speech_config=rc,
    audio_config=speechsdk.audio.AudioConfig(filename=str(PLAIN_WAV)),
    # microphone instead:
    # audio_config=speechsdk.audio.AudioConfig(use_default_microphone=True),
)

start = time.perf_counter()
result = recognizer.recognize_once_async().get()
print(f"[{(time.perf_counter() - start) * 1000:.0f} ms]  reason={result.reason}")

if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print("text:", result.text)
elif result.reason == speechsdk.ResultReason.NoMatch:
    print("no speech recognized:", result.no_match_details)
elif result.reason == speechsdk.ResultReason.Canceled:
    d = result.cancellation_details
    print("canceled:", d.reason, d.error_details)

### Continuous recognition

Event-driven, and the only correct choice for anything longer than one utterance.
The four events to know:

| Event | Meaning |
|---|---|
| `recognizing` | Partial hypothesis — render this live, do not store it |
| `recognized` | Final result for one utterance |
| `canceled` | Error or end of stream. **Auth failures arrive here** |
| `session_stopped` | Session ended |

The `canceled` handler is not optional. Without it, a bad key produces a recognizer
that returns nothing, raises nothing, and tells you nothing.

In [ ]:
import threading

# A longer file so continuous recognition has more than one utterance to find.
MULTI_WAV = OUT / "multi.wav"
sc = speech_config()
sc.speech_synthesis_voice_name = "en-GB-RyanNeural"
sc.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Riff16Khz16BitMonoPcm)
speechsdk.SpeechSynthesizer(
    speech_config=sc,
    audio_config=speechsdk.audio.AudioOutputConfig(filename=str(MULTI_WAV)),
).speak_text_async(
    "Hello, this is Priya Raman calling about order S O four four seven one. "
    "The Northwind X200 router arrived two weeks late. "
    "I was also charged twice on the card ending four two four two. "
    "Please call me back before Friday or I will escalate this."
).get()

rc = recognizer_config()
rc.speech_recognition_language = "en-GB"
cont = speechsdk.SpeechRecognizer(
    speech_config=rc,
    audio_config=speechsdk.audio.AudioConfig(filename=str(MULTI_WAV)),
)

utterances, partials = [], []
done = threading.Event()

cont.recognizing.connect(lambda evt: partials.append(evt.result.text))
cont.recognized.connect(
    lambda evt: utterances.append(evt.result.text)
    if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech
    else None
)
cont.session_stopped.connect(lambda evt: done.set())
cont.canceled.connect(
    lambda evt: (print("canceled:", evt.cancellation_details.reason,
                       evt.cancellation_details.error_details), done.set())
)

cont.start_continuous_recognition_async().get()
done.wait(timeout=60)
cont.stop_continuous_recognition_async().get()

print(f"{len(partials)} partial hypotheses, {len(utterances)} final utterances\n")
for i, u in enumerate(utterances, 1):
    print(f"  {i}. {u}")

TRANSCRIPT = " ".join(utterances)

## 5. Improving recognition without training: phrase lists

Before anyone proposes Custom Speech, try the free option. `PhraseListGrammar`
attaches hint terms to a single recognizer at runtime — no training data, no
project, no deployment, no cost.

What it can fix: product names, people, menu options, jargon the base model has
never seen. What it **cannot** fix: accents, background noise, 8 kHz telephony
audio. Those are acoustic problems and need Custom Speech with audio training data.

In [ ]:
JARGON_WAV = OUT / "jargon.wav"
JARGON_TEXT = (
    "Please escalate the Contoso Fabrikam handoff and check the Northwind X200 "
    "telemetry against the Adventure Works baseline."
)

sc = speech_config()
sc.speech_synthesis_voice_name = "en-GB-RyanNeural"
sc.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Riff16Khz16BitMonoPcm)
speechsdk.SpeechSynthesizer(
    speech_config=sc,
    audio_config=speechsdk.audio.AudioOutputConfig(filename=str(JARGON_WAV)),
).speak_text_async(JARGON_TEXT).get()


def recognize_file(path, phrases=None):
    rc = recognizer_config()
    rc.speech_recognition_language = "en-GB"
    r = speechsdk.SpeechRecognizer(
        speech_config=rc,
        audio_config=speechsdk.audio.AudioConfig(filename=str(path)),
    )
    if phrases:
        grammar = speechsdk.PhraseListGrammar.from_recognizer(r)
        for p in phrases:
            grammar.addPhrase(p)
    return r.recognize_once_async().get()


print("without phrase list:")
print(" ", recognize_file(JARGON_WAV).text)

print("\nwith phrase list:")
print(" ", recognize_file(JARGON_WAV, [
    "Contoso", "Fabrikam", "Northwind X200", "Adventure Works", "handoff", "telemetry",
]).text)

Synthetic audio is clean, so the difference here may be small or absent — the
recognizer already hears the words clearly. On real telephony audio with an accent,
the same phrase list routinely moves word error rate by several points on exactly
the terms you care about.

### The customization ladder

| Rung | Fixes | Cost / gating |
|---|---|---|
| **Phrase list** | Known vocabulary | Free, per-recognizer, no setup |
| **Custom Speech** | Accents, noise, telephony, deep jargon | Training cost; a deployed endpoint **bills while it exists** |
| **Custom Neural Voice** | A specific brand voice | **Limited Access**: application, approval, and recorded consent from the voice talent |

A trained Custom Speech model is used by pointing the SDK at its endpoint — one
line, exactly like swapping a model deployment name:

```python
rc = recognizer_config()
rc.endpoint_id = "<Endpoint ID from Speech Studio>"
```

> **Exam note.** Custom Neural Voice is gated behind Microsoft's Limited Access
> policy and requires documented consent from the voice talent, including a recorded
> consent statement. Custom Neural Voice Lite lowers the *data* requirement, not the
> *approval* requirement. Any scenario where a team clones a voice next week without
> an approved application is wrong.

## 6. Audio straight into a model

The cascaded pipeline throws information away at step one. "Fine.", "Fine!", and a
sarcastic "fine…" are the same four characters in a transcript.
`gpt-4o-audio-preview` / `gpt-4o-mini-audio-preview` (**preview**, version
`2024-12-17`) accept audio as a content part on the ordinary `/chat/completions`
API, so the paralinguistic signal survives.

Requirements: API version `2025-01-01-preview` or later, audio as base64 with
`format` of `wav` or `mp3`, maximum 20 MB. Deploy the model and set
`MODEL_AUDIO` in `.env`; this section skips cleanly without it.

In [ ]:
AUDIO_MODEL = cfg.get("MODEL_AUDIO")

if not AUDIO_MODEL:
    print("MODEL_AUDIO not set in .env - skipping.")
    print("Deploy gpt-4o-mini-audio-preview in the portal, then add:")
    print("  MODEL_AUDIO=gpt-4o-mini-audio-preview")
else:
    encoded = base64.b64encode(MULTI_WAV.read_bytes()).decode()
    start = time.perf_counter()
    resp = chat_client().chat.completions.create(
        model=AUDIO_MODEL,
        modalities=["text"],
        messages=[
            {"role": "system", "content":
             "You analyse customer calls. Report: the caller's request, their emotional "
             "state, any urgency cues in delivery, and whether escalation is threatened. "
             "Note explicitly if the audio conveys something the words alone do not."},
            {"role": "user", "content": [
                {"type": "text", "text": "Analyse this call."},
                {"type": "input_audio", "input_audio": {"data": encoded, "format": "wav"}},
            ]},
        ],
    )
    print(f"[{(time.perf_counter() - start) * 1000:.0f} ms]")
    print(resp.choices[0].message.content)
    show_usage(resp)

In [ ]:
# Audio out as well: modalities ["text", "audio"] plus an audio config.
# Supported output voices for the audio models are alloy, echo, and shimmer.
if AUDIO_MODEL:
    resp = chat_client().chat.completions.create(
        model=AUDIO_MODEL,
        modalities=["text", "audio"],
        audio={"voice": "alloy", "format": "wav"},
        messages=[{"role": "user", "content":
                   "In one sentence, apologise for a late delivery and confirm a replacement is on the way."}],
    )
    audio_out = resp.choices[0].message.audio
    reply_path = OUT / "model_reply.wav"
    reply_path.write_bytes(base64.b64decode(audio_out.data))
    print("transcript:", audio_out.transcript)
    print("wrote     :", reply_path.name, f"{reply_path.stat().st_size:,} bytes")
    show_usage(resp)

Look at the token usage. Audio tokens are counted separately in
`usage.prompt_tokens_details` / `completion_tokens_details` and are substantially
more expensive than text tokens. A voice agent built entirely on audio-in/audio-out
costs materially more per turn than a cascaded one — that is a real architectural
input, not a footnote.

## 7. Speech translation

`TranslationRecognizer` does speech in → transcript **and** translations out in one
pass. Multiple targets per call. Set `speech_synthesis_voice_name` and it will also
produce translated audio, giving you speech-to-speech translation without wiring
three services together.

In [ ]:
try:
    tcfg = speechsdk.translation.SpeechTranslationConfig(
        token_credential=credential(), endpoint=CUSTOM_DOMAIN_ENDPOINT
    )
    auth_path = "entra"
except Exception:
    tcfg = speechsdk.translation.SpeechTranslationConfig(
        subscription=cfg.require("AZURE_SPEECH_KEY"), region=REGION
    )
    auth_path = "key"

tcfg.speech_recognition_language = "en-GB"
for target in ["de", "ja", "pt"]:
    tcfg.add_target_language(target)

translator = speechsdk.translation.TranslationRecognizer(
    translation_config=tcfg,
    audio_config=speechsdk.audio.AudioConfig(filename=str(PLAIN_WAV)),
)

start = time.perf_counter()
result = translator.recognize_once_async().get()
print(f"[{auth_path}] [{(time.perf_counter() - start) * 1000:.0f} ms] reason={result.reason}\n")

if result.reason == speechsdk.ResultReason.TranslatedSpeech:
    print("source:", result.text)
    for lang, text in result.translations.items():
        print(f"  {lang}: {text}")
elif result.reason == speechsdk.ResultReason.Canceled:
    d = result.cancellation_details
    print("canceled:", d.reason, d.error_details)

In [ ]:
# The LLM alternative: translate the transcript with register control that
# TranslationRecognizer has no API surface to express.
start = time.perf_counter()
resp = chat_client().chat.completions.create(
    model=cfg.require("MODEL_MINI"),
    temperature=0,
    messages=[
        {"role": "system", "content":
         "Translate the support transcript to Japanese using polite business keigo. "
         "Keep order references in Latin script. Return only the translation."},
        {"role": "user", "content": TRANSCRIPT},
    ],
)
print(f"[{(time.perf_counter() - start) * 1000:.0f} ms]")
print(resp.choices[0].message.content)
show_usage(resp)

| Requirement | Choose |
|---|---|
| Live captions in three languages, lowest latency | `TranslationRecognizer` |
| Translated **audio** out with no extra plumbing | `TranslationRecognizer` + `speech_synthesis_voice_name` |
| Enforced terminology at volume | STT → Translator with a Custom Translator model → TTS |
| Register, honorifics, cultural nuance | STT → LLM → TTS |
| Preserve the speaker's emotion in the translation | Audio-capable model or Realtime |

## 8. Cascaded vs speech-to-speech

Time the full cascade — STT, then the model, then TTS — and compare it with the
figures the Realtime API reports. This is the number behind the architecture
decision, and it is worth measuring in your own region rather than trusting a table.

In [ ]:
stages = {}

t0 = time.perf_counter()
rc = recognizer_config()
rc.speech_recognition_language = "en-GB"
stt = speechsdk.SpeechRecognizer(
    speech_config=rc,
    audio_config=speechsdk.audio.AudioConfig(filename=str(PLAIN_WAV)),
).recognize_once_async().get()
stages["1. speech to text"] = (time.perf_counter() - t0) * 1000

t1 = time.perf_counter()
reply = chat_client().chat.completions.create(
    model=cfg.require("MODEL_MINI"),
    temperature=0,
    messages=[
        {"role": "system", "content": "You are a phone support agent. Reply in one short sentence."},
        {"role": "user", "content": stt.text},
    ],
).choices[0].message.content
stages["2. model"] = (time.perf_counter() - t1) * 1000

t2 = time.perf_counter()
sc = speech_config()
sc.speech_synthesis_voice_name = "en-GB-SoniaNeural"
synth = speechsdk.SpeechSynthesizer(speech_config=sc, audio_config=None)
start_result = synth.start_speaking_text_async(reply).get()
audio_stream = speechsdk.AudioDataStream(start_result)
buf = bytes(32000)
audio_stream.read_data(buf)  # first chunk only - that is what the user hears
stages["3. text to speech (first byte)"] = (time.perf_counter() - t2) * 1000

print("heard :", stt.text)
print("reply :", reply)
print()
for stage, ms in stages.items():
    print(f"  {stage:<34} {ms:>7.0f} ms")
print(f"  {'TOTAL to first audio':<34} {sum(stages.values()):>7.0f} ms")

### The decision

| | Cascaded (what you just built) | Realtime API (preview) |
|---|---|---|
| Transport | Three request/response calls | One persistent WebSocket |
| Latency to first audio | ~1.5–3 s | ~300–800 ms |
| Prosody in | Discarded at the transcript | Preserved |
| Prosody out | Whatever SSML you write | Model-generated, natural |
| Barge-in / interruption | You build it | Server-side voice activity detection |
| Moderation | Inspect and gate the transcript at every stage | Configure on the session; no text checkpoint |
| Auditability | Full transcript at every hop | Partial |
| Cost per turn | Lowest | Highest (audio tokens both ways) |
| Swap models/voices independently | Yes | No |

> **Exam note.** The trade-off in one line: **cascaded buys control and
> auditability; speech-to-speech buys latency and prosody.** A regulated workflow
> that must log, moderate, and redact every utterance argues for cascaded even
> though it is slower. A consumer voice assistant where turn-taking is the product
> argues for Realtime. "Realtime because it is newer" is not an answer.

The Realtime API is **preview** and uses the same underlying GPT-4o audio model as
the completions API, tuned for low-latency bidirectional streaming. It is not
exercised here because a WebSocket session does not belong in a notebook — see
the [Realtime audio quickstart](https://learn.microsoft.com/azure/ai-foundry/openai/realtime-audio-quickstart).

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Prove the 15-second rule.** Synthesize a 40-second passage, then transcribe it
   with `recognize_once_async` and with continuous recognition. Compare word counts.
   What `reason` does the single-shot call return, and why is that dangerous?

2. **SSML for a confirmation flow.** Write SSML that reads back a UK phone number,
   an order reference, and a delivery date so that each is unambiguous when heard
   once over a phone line. Synthesize it, transcribe the result, and check whether
   the recognizer recovers the original values. Where does the round trip fail?

3. **Redact before you speak.** Combine this unit with 04.1: transcribe `multi.wav`,
   run `recognize_pii_entities` over the transcript, and synthesize a summary with
   the card number and phone number removed. Where in the pipeline must redaction
   happen so the raw values never reach the TTS service?

4. **Budget the architecture.** Using the timings and token counts this notebook
   printed, estimate cost and latency per turn for (a) cascaded, (b) audio-in with
   text out plus Speech TTS, (c) fully audio-in/audio-out. State your assumptions
   and say which you would ship for a 24/7 support line, and why.

In [ ]:
# Your work here.

## Cleanup

In [ ]:
for f in sorted(OUT.glob("*.wav")):
    print(f"removing {f.name} ({f.stat().st_size:,} bytes)")
    f.unlink()

print("\nSpeech and the audio models are consumption-priced - nothing here bills hourly.")
print("If you deployed a Custom Speech endpoint in Speech Studio, DELETE IT:")
print("  a deployed custom endpoint bills for as long as it exists, used or not.")

Next: [05.1 — Build retrieval and grounding pipelines](../../05_information_extraction/01_retrieval_and_grounding/README.md)